# install dependencies

In [53]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
import statsmodels.api as sm

# Phase1 Data Preparation

In [ ]:
# 1. read file
file_path = '../../result/occ/analysis/merged_occ_irf_trajectories.csv'
df = pd.read_csv(file_path)

# 2. filter for income outcome
df_income = df[df['outcome'] == 'income'].copy()

# 3. ensure horizon is numeric and sort by horizon
df_income['horizon'] = pd.to_numeric(df_income['horizon'])
df_income = df_income.sort_values('horizon').reset_index(drop=True)

# 4. identify group columns 
group_cols = [col for col in df_income.columns if col.startswith('group')]

# 5. calculate CIR_36 for each group
cir_36_dict = {}
for col in group_cols:
    irf_1_36 = df_income[df_income['horizon'].between(1, 36)][col]
    cir_36_dict[col] = irf_1_36.sum()

cir_36_series = pd.Series(cir_36_dict, name='CIR_36')

print(cir_36_series)

group1_Managerial                      -0.055501
group2_Professional_specialty          -0.047927
group3_High_tech                       -0.240286
group4_Sales                           -0.368369
group5_Administrative_support          -0.087689
group6_Service                         -0.042406
group7_Farming_forestry_construction   -0.175243
group8_Precision_production_repair     -0.360971
group9_Machine_operators_transport      0.011926
Name: CIR_36, dtype: float64


In [55]:
# 1. read mapping file
mapping_path = '../../result/mapping/mapping_done.xlsx'
df_map = pd.read_excel(mapping_path, sheet_name='Sheet1', usecols='A,E', header=0)
df_map.columns = ['occ1990', 'SOC-2018']

df_map['occ1990'] = pd.to_numeric(df_map['occ1990'], errors='coerce').astype('Int64')
df_map['SOC-2018'] = df_map['SOC-2018'].astype(str).str.strip()
df_map = df_map.dropna(subset=['occ1990', 'SOC-2018'])

# 2. OCC1990 -> Group (1~9) 

def occ1990_to_group(occ):
    if 3 <= occ <= 37: return 1
    elif 43 <= occ <= 200: return 2
    elif 203 <= occ <= 235: return 3
    elif 243 <= occ <= 283: return 4
    elif 303 <= occ <= 389: return 5
    elif 405 <= occ <= 469: return 6
    elif (473 <= occ <= 498) or (558 <= occ <= 599) or (614 <= occ <= 617): return 7
    elif (503 <= occ <= 549) or (628 <= occ <= 699): return 8
    elif (703 <= occ <= 799) or (803 <= occ <= 889): return 9
    return np.nan

df_map['group'] = df_map['occ1990'].apply(occ1990_to_group)

# Use the previously computed CIR_36 values for each group
group_to_cir_36 = {}
for idx, val in cir_36_series.items():
    match = re.search(r'group(\d+)', str(idx))
    if match:
        group_to_cir_36[int(match.group(1))] = val

df_map['CIR_36'] = df_map['group'].map(group_to_cir_36)

# 3. y vector construction
df_final = df_map.drop_duplicates(subset='SOC-2018', keep='first')
y_series = df_final.set_index('SOC-2018')['CIR_36'].dropna()

print(y_series.head())

SOC-2018
11-1011   -0.055501
11-3012   -0.055501
33-1012   -0.055501
11-3031   -0.055501
11-3111   -0.055501
Name: CIR_36, dtype: float64


In [56]:
def load_and_prepare_onet_data(
    y_series,
    file_name,
    data_path='../../data/ONET',
    usecols=[0, 1, 4, 5, 7],
    scale_id='LV'
):
    file_path = f"{data_path.rstrip('/\\')}/{file_name}"
    df = pd.read_excel(file_path, usecols=usecols, header=0)
    df.columns = ['SOC_Code', 'Sub_Code', 'Element_Name', 'Scale_ID', 'Data_Value']

    df['SOC_Code'] = df['SOC_Code'].astype(str).str.strip()
    df['Element_Name'] = df['Element_Name'].astype(str).str.strip()
    df['Scale_ID'] = df['Scale_ID'].astype(str).str.strip().str.upper()
    df['Sub_Code'] = df['Sub_Code'].astype(str).str.strip().str.zfill(2)

    soc_elem_means = df.groupby(['SOC_Code', 'Element_Name'])['Data_Value'].mean().reset_index()
    soc_elem_means.rename(columns={'Data_Value': 'Mean_Val'}, inplace=True)

    df = df.merge(soc_elem_means, on=['SOC_Code', 'Element_Name'], how='left')
    df.loc[df['Sub_Code'] == '00', 'Data_Value'] = df.loc[df['Sub_Code'] == '00', 'Mean_Val']

    df = df[df['Sub_Code'] == '00'].copy()
    df.drop(columns=['Mean_Val', 'Sub_Code'], inplace=True)

    df = df[df['Scale_ID'] == scale_id].copy()
    df = df.dropna(subset=['Data_Value'])

    df_wide = df.pivot_table(
        index='SOC_Code',
        columns='Element_Name',
        values='Data_Value',
        aggfunc='mean'
    )
    df_wide = df_wide.astype(float)
    df_wide = df_wide.fillna(df_wide.median())

    if df_wide.shape[1] == 0:
        raise ValueError("df_wide has no columns; check Sub_Code and Scale_ID filters above.")

    scaler = StandardScaler()
    X_df = pd.DataFrame(
        scaler.fit_transform(df_wide),
        columns=df_wide.columns,
        index=df_wide.index
    )

    aligned_idx = X_df.index.intersection(y_series.index)
    X = X_df.loc[aligned_idx].values
    y_aligned = y_series.loc[aligned_idx].values

    return X_df, aligned_idx, X, y_aligned

# Phase2: LASSO Estimation

In [57]:
def run_lasso_selection(
    X,
    y_aligned,
    feature_names,
    top_n=10,
    n_splits=10,
    random_state=42,
    max_iter=5000
):
    feature_names = pd.Index(feature_names)
    cv_strategy = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    lasso_cv = LassoCV(
        alphas=None,
        cv=cv_strategy,
        max_iter=max_iter,
        random_state=random_state,
        n_jobs=-1
    )
    lasso_cv.fit(X, y_aligned)

    best_alpha = lasso_cv.alpha_
    best_coefs = lasso_cv.coef_
    cv_mse_path = lasso_cv.mse_path_
    cv_mean_mse = cv_mse_path.mean(axis=1)

    coef_abs = np.abs(best_coefs)
    top_idx = np.argsort(coef_abs)[::-1][:top_n]
    top_names = feature_names.take(top_idx).tolist()
    top_coefs = best_coefs[top_idx]

    selected_mask = np.zeros(len(feature_names), dtype=bool)
    selected_mask[top_idx] = True

    return {
        'lasso_cv': lasso_cv,
        'best_alpha': best_alpha,
        'best_coefs': best_coefs,
        'cv_mse_path': cv_mse_path,
        'cv_mean_mse': cv_mean_mse,
        'feature_names': feature_names,
        'top_idx': top_idx,
        'top_names': top_names,
        'top_coefs': top_coefs,
        'selected_mask': selected_mask
    }

In [58]:
def calculate_post_lasso_r2(X, y_aligned, selected_mask, selected_feature_names):
    X_selected = X[:, selected_mask]
    X_selected_const = sm.add_constant(X_selected)

    ols_model = sm.OLS(y_aligned, X_selected_const).fit(cov_type='HC3')
    r_squared = ols_model.rsquared
    r_squared_adj = ols_model.rsquared_adj
    ci = ols_model.conf_int()

    ols_results_df = pd.DataFrame({
        'O*NET_Activity': selected_feature_names,
        'OLS_Coefficient': ols_model.params[1:],
        'CI_Lower': ci[1:, 0],
        'CI_Upper': ci[1:, 1],
        'Std_Error': ols_model.bse[1:],
        'P_Value': ols_model.pvalues[1:],
        'Sig_10%': ols_model.pvalues[1:] < 0.10,
        'Sig_5%': ols_model.pvalues[1:] < 0.05
    }).sort_values('OLS_Coefficient', key=abs, ascending=False)

    return {
        'ols_model': ols_model,
        'r_squared': r_squared,
        'r_squared_adj': r_squared_adj,
        'ols_results_df': ols_results_df
    }

# Main

In [17]:
X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data(
    y_series,
    file_name='Work Activities.xlsx'
)

lasso_results = run_lasso_selection(
    X,
    y_aligned,
    X_df.loc[aligned_idx].columns
)

post_lasso_results = calculate_post_lasso_r2(
    X,
    y_aligned,
    lasso_results['selected_mask'],
    lasso_results['top_names']
)
print("Selected O*NET Activities and their coefficients:")
print(post_lasso_results['ols_results_df'][['O*NET_Activity', 'OLS_Coefficient', 'CI_Lower', 'CI_Upper', 'P_Value', 'Sig_5%']])
print(f"R-squared: {post_lasso_results['r_squared']:.4f}")
print(f"Adjusted R-squared: {post_lasso_results['r_squared_adj']:.4f}")

Selected O*NET Activities and their coefficients:
                                      O*NET_Activity  OLS_Coefficient  \
6                                Getting Information        -0.055715   
5                      Staffing Organizational Units        -0.032225   
7              Updating and Using Relevant Knowledge        -0.030822   
9                 Controlling Machines and Processes        -0.029743   
2        Organizing, Planning, and Prioritizing Work         0.028881   
4               Developing Objectives and Strategies         0.022578   
1                      Selling or Influencing Others         0.020734   
3                      Developing and Building Teams         0.019321   
8  Communicating with Supervisors, Peers, or Subo...         0.017102   
0     Repairing and Maintaining Mechanical Equipment         0.014868   

   CI_Lower  CI_Upper       P_Value  Sig_5%  
6 -0.077213 -0.034217  3.783381e-07    True  
5 -0.049453 -0.014996  2.465149e-04    True  
7 -0.043

In [59]:
X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data(
    y_series,
    file_name='Skills.xlsx'
)

lasso_results = run_lasso_selection(
    X,
    y_aligned,
    X_df.loc[aligned_idx].columns
)

post_lasso_results = calculate_post_lasso_r2(
    X,
    y_aligned,
    lasso_results['selected_mask'],
    lasso_results['top_names']
)
print("Selected O*NET Activities and their coefficients:")
print(post_lasso_results['ols_results_df'][['O*NET_Activity', 'OLS_Coefficient', 'CI_Lower', 'CI_Upper', 'P_Value', 'Sig_5%']])
print(f"R-squared: {post_lasso_results['r_squared']:.4f}")
print(f"Adjusted R-squared: {post_lasso_results['r_squared_adj']:.4f}")

Selected O*NET Activities and their coefficients:
            O*NET_Activity  OLS_Coefficient  CI_Lower  CI_Upper   P_Value  \
6    Operations Monitoring        -0.046303 -0.067717 -0.024888  0.000023   
0               Persuasion         0.033390  0.013102  0.053679  0.001257   
3              Mathematics         0.031178  0.015235  0.047121  0.000127   
5  Complex Problem Solving        -0.026104 -0.045375 -0.006834  0.007930   
1                Repairing        -0.020075 -0.035697 -0.004453  0.011783   
2             Installation        -0.019897 -0.031237 -0.008558  0.000583   
8      Service Orientation        -0.014904 -0.026830 -0.002978  0.014313   
7          Time Management        -0.011496 -0.024662  0.001670  0.087017   
4               Monitoring         0.009832 -0.006607  0.026271  0.241109   
9        Technology Design        -0.009252 -0.022954  0.004450  0.185692   

   Sig_5%  
6    True  
0    True  
3    True  
5    True  
1    True  
2    True  
8    True  
7   Fa

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


In [60]:
X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data(
    y_series,
    file_name='Knowledge.xlsx'
)

lasso_results = run_lasso_selection(
    X,
    y_aligned,
    X_df.loc[aligned_idx].columns
)

post_lasso_results = calculate_post_lasso_r2(
    X,
    y_aligned,
    lasso_results['selected_mask'],
    lasso_results['top_names']
)
print("Selected O*NET Activities and their coefficients:")
print(post_lasso_results['ols_results_df'][['O*NET_Activity', 'OLS_Coefficient', 'CI_Lower', 'CI_Upper', 'P_Value', 'Sig_5%']])
print(f"R-squared: {post_lasso_results['r_squared']:.4f}")
print(f"Adjusted R-squared: {post_lasso_results['r_squared_adj']:.4f}")

Selected O*NET Activities and their coefficients:
                  O*NET_Activity  OLS_Coefficient  CI_Lower  CI_Upper  \
5                    Mathematics        -0.074597 -0.092524 -0.056671   
8               English Language         0.034914  0.023200  0.046628   
1     Public Safety and Security         0.027652  0.015315  0.039988   
6  Customer and Personal Service         0.027219  0.013776  0.040663   
0                     Mechanical        -0.026474 -0.038781 -0.014167   
2            Sales and Marketing         0.024282  0.011499  0.037064   
4                        Physics        -0.023074 -0.035083 -0.011065   
9             Law and Government        -0.020232 -0.033236 -0.007228   
3       Economics and Accounting        -0.017182 -0.030098 -0.004266   
7      Production and Processing         0.013598 -0.000201  0.027396   

        P_Value  Sig_5%  
5  3.466437e-16    True  
8  5.160935e-09    True  
1  1.116877e-05    True  
6  7.236420e-05    True  
0  2.483774e-05 

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(
